In [6]:
%reload_ext autoreload
%autoreload 2

In [39]:
# --------------------- Import Libraries ---------------------
import os
import copy
import sys
import yaml
import pickle
from pathlib import Path
from dataclasses import dataclass

import scanpy as sc
import anndata as ad
import squidpy as sq
import scvi

import numpy as np
import networkx as nx
from sklearn.neighbors import NearestNeighbors
import scipy.sparse as sp
from scipy.stats import pearsonr

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import pytorch_lightning as pl
import torch_geometric.transforms as T
from torch_geometric.data import Batch
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_adj
from torch_geometric.loader import DataLoader as BatchBuilder
from torch_geometric.utils import is_undirected
from torch_geometric.utils.convert import from_scipy_sparse_matrix

# --------------------- Display Settings ---------------------
# display setting all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [33]:
from vqniche.metrics.pearson_correlation import pearson_correlation
from vqniche.metrics.mmd import compute_mmd_score
from vqniche.preprocessors.graph_constructors import set_edge_index_name, spatial_neighbors

In [41]:
def build_edge_index(
    train_adata,
    test_adata
):
    adata = ad.concat([train_adata, test_adata])
    edge_index_name = "8-nn_edge_index"
    adata = spatial_neighbors(
                    adata,
                    coord_type="generic",
                    spatial_key="spatial",
                    delaunay="False",
                    n_neighs=8,
                    radius=None,
                    include_self_loop=True,
                    key_added=edge_index_name
                )
    connectivity_matrix = adata.obsp[f"{edge_index_name}_connectivities"]
    count_nnz = (connectivity_matrix != connectivity_matrix.T).nnz
    if count_nnz > 0:
        print(f"{edge_index_name} is not symmetric. Making it symmetric...")
        adata.obsp[f"{edge_index_name}_connectivities"] = (
            adata.obsp[f"{edge_index_name}_connectivities"].maximum(
                adata.obsp[f"{edge_index_name}_connectivities"].T
            )
        )

    edge_index = from_scipy_sparse_matrix(
                    adata.obsp[f"{edge_index_name}_connectivities"]
                )[0]    
    return edge_index

In [13]:
def train_scvi_model(
    train_adata,
    layer_name: str = "counts",
    batch_key: str = "batch",
    max_epochs: int = 50,
    **train_kwargs,
):
    """
    Train an SCVI model on train_adata and return the trained model.
    """
    scvi.model.SCVI.setup_anndata(
        train_adata,
        layer=layer_name,
        batch_key=batch_key,
    )

    model = scvi.model.SCVI(train_adata)
    model.train(max_epochs=max_epochs, **train_kwargs)
    return model

In [19]:
def spatial_impute_test_from_model(
    model,
    train_adata,
    test_adata,
    k: int = 8,
    sigma: float | None = None,
):
    """
    Use a trained SCVI model (fit on train_adata) to spatially impute
    gene expression for test_adata.
    """
    # Estimate denoised counts
    train_mat = model.get_normalized_expression().values

    nn = NearestNeighbors(n_neighbors=k)
    nn.fit(train_adata.obsm["spatial"])

    dists, idx = nn.kneighbors(test_adata.obsm["spatial"])

    # --- convert distances to weights (Gaussian kernel) ---
    eps = 1e-8
    if sigma is None:
        nonzero = dists[dists > 0]
        if nonzero.size == 0:
            sigma = 1.0
        else:
            sigma = float(np.median(nonzero))
    sigma = max(sigma, eps)

    weights = np.exp(-(dists ** 2) / (2.0 * sigma ** 2))
    weights = weights / (weights.sum(axis=1, keepdims=True) + eps)  # normalize per test cell

    # Compute weighted average of neighbor expressions
    neighbor_expr = train_mat[idx, :] # shape: (n_test, k, n_genes)
    imputed_test = (weights[..., None] * neighbor_expr).sum(axis=1)  # shape: (n_test, n_genes)
    
    # Scale by empirical read depth
    imputed_test_counts = imputed_test * test_adata.X.sum(axis=1).reshape(-1,1)    

    # Add back to adata
    if sp.issparse(test_adata.X):
        X_np = test_adata.X.toarray()
    else:
        X_np = np.asarray(test_adata.X)

    test_adata.uns["X"] = torch.from_numpy(X_np.astype("float32"))
    test_adata.uns["X_hat"] = torch.from_numpy(imputed_test.astype("float32"))

    return test_adata

# mmb0-4b_1p

In [11]:
train_adata = sc.read_h5ad("/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/reproducibility/analysis/notebooks/mmb0-4b_1p_train.h5ad")
train_adata

AnnData object with n_obs × n_vars = 47313 × 1000
    obs: 'batch', 'cell_type'
    obsm: 'spatial'

In [17]:
test_adata = sc.read_h5ad("/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/reproducibility/analysis/notebooks/mmb0-4b_1p_test.h5ad")
test_adata

AnnData object with n_obs × n_vars = 843 × 1000
    obs: 'batch', 'cell_type'
    obsm: 'spatial'

In [42]:
edge_index = build_edge_index(
    train_adata,
    test_adata
)
edge_index

tensor([[    0,     0,     0,  ..., 48155, 48155, 48155],
        [  231,   219,     2,  ..., 46366, 46482, 48155]])

In [43]:
edge_index.shape

torch.Size([2, 337010])

In [35]:
adata

AnnData object with n_obs × n_vars = 48156 × 1000
    obs: 'batch', 'cell_type'
    uns: '8-nn_edge_index_neighbors'
    obsm: 'spatial'
    obsp: '8-nn_edge_index_connectivities', '8-nn_edge_index_distances'

In [14]:
# Train
model = train_scvi_model(train_adata)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA A100-SXM4-80GB MIG 1c.2g.20gb') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [MIG-GPU-65984e58-2b6b-21f8-41cd-7854ff83d259/3/0]
/software/cellgen/team361/am84/envs/vqniche-reproducibility/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Epoch 50/50: 100%|██████████| 50/50 [01:47<00:00,  2.14s/it, v_num=1, train_loss_step=299, train_loss_epoch=293]

`Trainer.fit` stopped: `max_epochs=50` reached.


Epoch 50/50: 100%|██████████| 50/50 [01:47<00:00,  2.15s/it, v_num=1, train_loss_step=299, train_loss_epoch=293]


In [20]:
# Impute
test_adata = spatial_impute_test_from_model(
                model,
                train_adata,
                test_adata,
                k=8
            )
test_adata

AnnData object with n_obs × n_vars = 843 × 1000
    obs: 'batch', 'cell_type'
    uns: 'X', 'X_hat'
    obsm: 'spatial'

In [22]:
pearson_correlation(
    X=test_adata.uns['X'],
    X_hat=test_adata.uns['X_hat'],
    mean=True
)

# SQUINT = (0.6652+0.6647+0.6650)/3 = 0.6649

0.65503436

In [27]:
# Compute sums before division
X_sum = test_adata.uns['X'].sum(axis=1, keepdims=True)
X_hat_sum = test_adata.uns['X_hat'].sum(axis=1, keepdims=True)
D = (test_adata.uns['X'] / X_sum).numpy()
D_hat = (test_adata.uns['X_hat'] / X_hat_sum).numpy()
compute_mmd_score(
            D=[row for row in D],
            D_hat=[row for row in D_hat],
            method='scipy',
            kernel='l1_gaussian_tv',
        )

0.14933286708523277

# xhk1020-CV1-CV2-5b_1p

In [44]:
train_adata = sc.read_h5ad("/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/reproducibility/analysis/notebooks/xhk1020-CV1-CV2-5b_1p_train.h5ad")
train_adata

AnnData object with n_obs × n_vars = 257746 × 1000
    obs: 'batch', 'cell_type'
    obsm: 'spatial'

In [45]:
test_adata = sc.read_h5ad("/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/reproducibility/analysis/notebooks/xhk1020-CV1-CV2-5b_1p_test.h5ad")
test_adata

AnnData object with n_obs × n_vars = 2447 × 1000
    obs: 'batch', 'cell_type'
    obsm: 'spatial'

In [46]:
# Train
model = train_scvi_model(train_adata)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [MIG-GPU-65984e58-2b6b-21f8-41cd-7854ff83d259/3/0]
/software/cellgen/team361/am84/envs/vqniche-reproducibility/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Epoch 50/50: 100%|██████████| 50/50 [09:43<00:00, 11.64s/it, v_num=1, train_loss_step=433, train_loss_epoch=442]

`Trainer.fit` stopped: `max_epochs=50` reached.


Epoch 50/50: 100%|██████████| 50/50 [09:43<00:00, 11.67s/it, v_num=1, train_loss_step=433, train_loss_epoch=442]


In [47]:
# Impute
test_adata = spatial_impute_test_from_model(
                model,
                train_adata,
                test_adata,
                k=8
            )
test_adata

AnnData object with n_obs × n_vars = 2447 × 1000
    obs: 'batch', 'cell_type'
    uns: 'X', 'X_hat'
    obsm: 'spatial'

In [48]:
pearson_correlation(
    X=test_adata.uns['X'],
    X_hat=test_adata.uns['X_hat'],
    mean=True
)

# SQUINT = (0.5085 + 0.5019 + 0.4974) / 3 = 0.5026

0.0942983

# xhs1000-39b_1p-oriented-7

In [49]:
train_adata = sc.read_h5ad("/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/reproducibility/analysis/notebooks/xhs1000-39b_1p-oriented-7_train.h5ad")
train_adata

AnnData object with n_obs × n_vars = 109547 × 1000
    obs: 'batch', 'cell_type'
    obsm: 'spatial'

In [50]:
test_adata = sc.read_h5ad("/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/reproducibility/analysis/notebooks/xhs1000-39b_1p-oriented-7_test.h5ad")
test_adata

AnnData object with n_obs × n_vars = 731 × 1000
    obs: 'batch', 'cell_type'
    obsm: 'spatial'

In [51]:
# Train
model = train_scvi_model(train_adata)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [MIG-GPU-65984e58-2b6b-21f8-41cd-7854ff83d259/3/0]
/software/cellgen/team361/am84/envs/vqniche-reproducibility/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Epoch 50/50: 100%|██████████| 50/50 [04:08<00:00,  4.97s/it, v_num=1, train_loss_step=225, train_loss_epoch=188]

`Trainer.fit` stopped: `max_epochs=50` reached.


Epoch 50/50: 100%|██████████| 50/50 [04:08<00:00,  4.98s/it, v_num=1, train_loss_step=225, train_loss_epoch=188]


In [52]:
# Impute
test_adata = spatial_impute_test_from_model(
                model,
                train_adata,
                test_adata,
                k=8
            )
test_adata

AnnData object with n_obs × n_vars = 731 × 1000
    obs: 'batch', 'cell_type'
    uns: 'X', 'X_hat'
    obsm: 'spatial'

In [53]:
pearson_correlation(
    X=test_adata.uns['X'],
    X_hat=test_adata.uns['X_hat'],
    mean=True
)

0.056197446